# 04 - Final evaluation (skill 8)
**One** official run on the locked test splits for every policy, then comparison, plots, failure
analysis and cost. Part A needs a GPU (policies run on `test`, `test_injection`, `test_ood`); Part B
runs on the laptop from the results pulled off the HF dataset repo.

Rules: the test splits are evaluated once. `FINAL_RUN.json` locks the run; a re-run needs
`--rerun-reason`, a DECISIONS.md entry, and re-running **all** policies. Fill the `<user>`
placeholders in `configs/final.yaml` with your adapter ids first (the script refuses otherwise).

## Part A - GPU session

In [ ]:
# A1. GPU + parameters
!nvidia-smi
import os, sys
REPO_URL = "https://github.com/YOUR_GITHUB_USER/tinyops"
HF_USER = "YOUR_HF_USERNAME"
DATA_REPO = f"{HF_USER}/tinyops-data"

In [ ]:
# A2. install, secrets, environment record, hash check (aborts on any mismatch)
if not os.path.exists("tinyops"):
    !git clone {REPO_URL} tinyops
%cd tinyops
if sys.version_info >= (3, 12):
    PY = "python"
    !pip install -q -e ".[train]"
else:
    PY = "/content/venv/bin/python"
    !pip -q install uv
    !uv venv --python 3.12 /content/venv
    !uv pip install --python {PY} -q -e ".[train]"
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
!{PY} -m tinyops.utils.envinfo
!{PY} -m tinyops.scenarios.build_dataset verify

In [ ]:
# A3. bring in the skill-5 baseline episodes and the Gemini results (they are reused when the prompt
#     version, seed and step limit match; otherwise the policy is re-run)
from huggingface_hub import snapshot_download
snapshot_download(DATA_REPO, repo_type="dataset", local_dir="hf_data", allow_patterns=["baselines/*"])
!mkdir -p results/episodes results/runs && cp -n hf_data/baselines/episodes/* results/episodes/ && cp -n hf_data/baselines/runs/* results/runs/

In [ ]:
# A4. the final run. It is resumable per episode; to split it across sessions use --only (the lock
#     file is written only when every policy x split exists).
!{PY} -m tinyops.eval.final_eval --config configs/final.yaml

In [ ]:
# A5. AuditKIT cross-check (agent_eval) and the supplementary single-turn red team, base vs final
for pol in ["base_zeroshot", "final"]:
    for split in ["test", "test_injection", "test_ood"]:
        !{PY} -m tinyops.eval.crosscheck --episodes results/final/episodes/{pol}_{split}.jsonl --split {split} --run {pol}_{split} --out-dir results/final/auditkit
!{PY} -m tinyops.eval.redteam --model CohereLabs/tiny-aya-global --name base_zeroshot
!{PY} -m tinyops.eval.redteam --model CohereLabs/tiny-aya-global --adapter "<sft adapter>+<stage B adapter>" --name final

In [ ]:
# A6. push the locked results to the HF dataset repo (folder final/)
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(DATA_REPO, repo_type="dataset", exist_ok=True)
api.upload_folder(folder_path="results/final", path_in_repo="final", repo_id=DATA_REPO, repo_type="dataset",
                  ignore_patterns=["plots/*", "failures/*"])

## Part B - laptop (CPU)
Pull `final/` from the HF dataset repo into `results/final/`, then:
```
python -m tinyops.eval.compare                     # report.json + tables.md
python -m tinyops.report.make_plots                # 5 figures (PNG + SVG)
python -m tinyops.report.failure_analysis sample   # renders + draft labels -> results/final/failures/
#   -> open labels.csv, read each rendered failure, fill confirmed_label (one of the 8 categories)
python -m tinyops.report.failure_analysis doc      # docs/FAILURE_ANALYSIS.md (says DRAFT until all rows are confirmed)
python -m tinyops.report.roi                       # needs the cited prices in configs/roi.yaml
python -m tinyops.report.make_tables --readme      # README results table, straight from report.json
```
Every number planned for the README/article must trace to a file in `results/final/`.